In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim ,length
from pyspark.sql.types import StringType
from pyspark.sql.types import DateType

In [0]:
df=spark.table("bronze.erp_loc_a101")
display(df)

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df=df.withColumn(field.name,trim(col(field.name)))

In [0]:
df = df.withColumn("cid", F.regexp_replace(col("cid"), "-", ""))

In [0]:
df = df.withColumn(
    "cntry",
    F.when(col("cntry") == "DE", "Germany")
     .when(col("cntry").isin("US", "USA"), "United States")
     .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
     .otherwise(col("cntry")))


In [0]:

RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.show(5)

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver.erp_loc_a101")

In [0]:
%sql
select * from silver.erp_loc_a101 limit 5

In [0]:
%sql
drop table if exists silver.erp_loc
